# CS383: Data Science and Machine Learning
## Lecture 9 — Evaluation Metrics for Classification

*Dr. Thitima Srivatanakul*

### Guiding question
**A classifier claims 95% accuracy. Is that good? What if 95% of cases already belong to one class —
could a model that never even looks at the data hit that same number?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain why accuracy alone can be misleading on imbalanced data, and check it against the
  majority-class baseline
- build a confusion matrix (TP/FP/FN/TN) for a binary classifier and read what each cell means
- compute precision, recall, and F1, and explain the tradeoff between them in terms of the real-world
  cost of a false positive vs. a false negative
- move the classification threshold away from 0.5 and explain how that trades recall for precision
- compute and interpret a ROC curve and AUC as a threshold-independent measure of ranking quality
- apply all of the above to a real, imbalanced NYC 311 classification problem

### Where this fits
Lecture 8 gave you your first classifiers and a first taste of "accuracy vs. baseline." This lecture is
entirely about what to measure once you have a classifier — especially when, like most real
classification problems, the classes aren't close to 50/50.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. Twenty transactions, two of them fraud — you'll see a "do-nothing" classifier
score 90% accuracy without learning a thing, meet the confusion matrix, drag a threshold slider and watch
precision and recall trade off in real time, and build a ROC curve one threshold at a time.

**[Open the "Precision, Recall, and the Cost of Being Wrong" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect09/evaluation_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Why Accuracy Lies

Twenty credit card transactions, two of them fraudulent — a realistic imbalance for this kind of problem.
Watch what "accuracy" does with that imbalance before we've fit anything at all.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
    f1_score, classification_report, precision_recall_curve, roc_curve, roc_auc_score,
)

# 20 transactions, 2 fraudulent (indices 3 and 16) -- an illustrative 10% fraud rate.
TRUE_FRAUD = np.array([0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0])

# A toy risk model's predicted P(fraud) for each transaction -- illustrative numbers, built to teach
# the mechanics cleanly. Real data comes in Part 5.
RISK_SCORE = np.array([0.05,0.08,0.12,0.62,0.15,0.22,0.09,0.55,0.18,0.31,
                        0.44,0.07,0.28,0.19,0.11,0.36,0.41,0.24,0.33,0.14])

print(f"{TRUE_FRAUD.sum()} fraudulent transactions out of {len(TRUE_FRAUD)} ({TRUE_FRAUD.mean():.0%})")

In [ ]:
# The laziest possible classifier: predict "not fraud" for absolutely everything.
lazy_predictions = np.__________(len(TRUE_FRAUD), dtype=int)

lazy_accuracy = (lazy_predictions == TRUE_FRAUD).mean()
print(f"Accuracy of a classifier that never looks at the data: {lazy_accuracy:.1%}")

**90% accuracy, from a model that does nothing.** That's the majority-class baseline you met in Lecture
8, just more dramatic here because the imbalance is sharper. Any real fraud model has to clear *this* bar
to prove it learned something — not clear 50%, and not be judged on accuracy alone, since accuracy alone
can't even tell the lazy classifier apart from a genuinely useful one yet.

### The confusion matrix

Apply the usual 0.5 threshold to `RISK_SCORE` and every prediction falls into one of four buckets:

| | Predicted: not fraud | Predicted: fraud |
|---|---|---|
| **Actual: not fraud** | True Negative (TN) | False Positive (FP) |
| **Actual: fraud** | False Negative (FN) | True Positive (TP) |

- **TP**: caught actual fraud. **TN**: correctly cleared a real transaction.
- **FP**: a false alarm — flagged a legitimate transaction as fraud.
- **FN**: a miss — actual fraud that slipped through uncaught.

Accuracy only looks at the diagonal (TP + TN) against everything. It never asks *which* kind of mistake
the off-diagonal cells represent — and for a rare, costly event like fraud, that distinction is the whole
point.

In [ ]:
predictions_50 = (RISK_SCORE >= 0.5).astype(int)

cm = __________(TRUE_FRAUD, predictions_50)
ConfusionMatrixDisplay(cm, display_labels=["not fraud", "fraud"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion Matrix (threshold = 0.5)")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

One fraud caught (TP=1), one fraud missed (FN=1), one false alarm (FP=1), seventeen correctly cleared
(TN=17). Accuracy on *this* model: (1+17)/20 = 90% — identical to the lazy classifier's accuracy, even
though this one clearly did something the lazy one didn't (it gave the actual fraud cases meaningfully
higher scores: 0.62 and 0.41, vs. everyone else mostly under 0.36). Accuracy can't see that difference.
The confusion matrix can.

---

## Part 2 — Precision, Recall, and F1

Two questions the confusion matrix's four numbers can answer, each asking something different:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

- **Precision**: "Of everything I flagged as fraud, how much actually was?" High precision means few
  false alarms.
- **Recall**: "Of all the actual fraud, how much did I catch?" High recall means few missed cases.

In [ ]:
precision_byhand = tp / (tp + fp)
recall_byhand = tp / (tp + __________)

print(f"Precision: {precision_byhand:.3f}  ({tp} caught out of {tp+fp} flagged)")
print(f"Recall:    {recall_byhand:.3f}  ({tp} caught out of {tp+fn} actual fraud)")

In [ ]:
precision = precision_score(TRUE_FRAUD, predictions_50)
recall = recall_score(TRUE_FRAUD, predictions_50)
f1 = __________(TRUE_FRAUD, predictions_50)

print(f"sklearn precision: {precision:.3f}")
print(f"sklearn recall:    {recall:.3f}")
print(f"sklearn F1:        {f1:.3f}")

### F1: one number when you need to balance both

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The *harmonic* mean of precision and recall, not the plain average — it punishes a big gap between them
more than a plain average would. A model with precision 1.0 and recall 0.01 has an *average* of about
0.50, but an F1 near 0.02: F1 refuses to reward a model that's excellent on one metric and useless on the
other.

### Which one matters more? Depends on the cost of being wrong

- **Fraud detection**: missing real fraud (low recall) costs real money. A false alarm (low precision)
  just means a customer gets an extra verification text. Recall usually matters more here.
- **Spam filtering**: a false alarm (low precision) buries a real email in spam, maybe an important one.
  Missing a spam email (low recall) is just mildly annoying. Precision usually matters more here.

There's no universal right answer — it's a business/domain decision, not a math one. The math (precision,
recall, F1) just gives you the vocabulary to make that decision explicitly instead of by accident.

In [ ]:
print(__________(TRUE_FRAUD, predictions_50, target_names=["not fraud", "fraud"]))

`classification_report()` is a shortcut that prints precision, recall, and F1 for *both* classes at once
(plus `support`, how many actual cases of each class there were) — usually faster than calling three
separate functions.

---

## Part 3 — Threshold Tuning

`.predict()` applies a 0.5 cutoff to `.predict_proba()` by default — but nothing about 0.5 is special.
Lowering the threshold flags more transactions as fraud: recall can only go up (or stay the same), while
precision usually pays for it.

In [ ]:
for threshold in [0.5, 0.4, 0.2]:
    preds = (RISK_SCORE >= threshold).astype(int)
    p = precision_score(TRUE_FRAUD, preds, zero_division=0)
    r = __________(TRUE_FRAUD, preds)
    n_flagged = preds.sum()
    print(f"threshold={threshold:.1f}: {n_flagged:>2} flagged -> precision={p:.3f}, recall={r:.3f}")

At 0.5, recall is 0.5 — the model catches one of the two fraud cases. Drop the threshold to 0.4 and
recall jumps to 1.0 (both frauds now caught), at the cost of flagging two innocent transactions along the
way. Drop it further to 0.2 and recall stays at 1.0 (you already caught everything there was to catch),
but precision keeps falling as more and more innocent transactions get swept in. This is the tradeoff,
laid out as numbers instead of just a concept.

In [ ]:
precisions, recalls, thresholds = __________(TRUE_FRAUD, RISK_SCORE)

plt.plot(thresholds, precisions[:-1], label="Precision", color="#2B6CB0")
plt.plot(thresholds, recalls[:-1], label="Recall", color="#D98C3F")
plt.axvline(0.5, color="#9AA5B1", linestyle="--", linewidth=1, label="default threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision & Recall vs. Threshold")
plt.legend()
plt.show()

`precision_recall_curve()` computes precision and recall at *every* threshold implied by the data, in
one call — the three-threshold sweep above was the same idea, done by hand at just three points. Reading
this chart left to right (low threshold to high): recall starts high and only ever drops or holds steady;
precision generally rises but can wobble. Picking a threshold means picking a point on this chart that
matches how costly each kind of mistake actually is for the problem at hand.

---

## Part 4 — ROC Curve and AUC

A different way to look at the same tradeoff, using two rates instead of precision/recall:

$$TPR = \text{Recall} = \frac{TP}{TP+FN} \qquad FPR = \frac{FP}{FP+TN}$$

**TPR** (true positive rate) is just recall again. **FPR** (false positive rate) asks: of everyone who
was *actually* innocent, what fraction did the model wrongly flag? A **ROC curve** plots FPR (x-axis)
against TPR (y-axis) as the threshold sweeps from high to low.

In [ ]:
fpr, tpr, roc_thresholds = __________(TRUE_FRAUD, RISK_SCORE)

plt.plot(fpr, tpr, color="#2B6CB0", linewidth=2, label="This model")
plt.plot([0, 1], [0, 1], color="#9AA5B1", linestyle="--", label="Random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

A model that ranks purely by chance traces the diagonal line — at any threshold, its TPR and FPR rise
together at the same rate. A useful model bows up and to the left of that diagonal: it can achieve a high
TPR while keeping FPR low, because it's genuinely separating the two classes, not just guessing.

In [ ]:
auc = __________(TRUE_FRAUD, RISK_SCORE)
print(f"AUC: {auc:.3f}")

**AUC** (area under the ROC curve) is exactly what it sounds like — 1.0 is a perfect ranker, 0.5 is random
guessing. Unlike accuracy, precision, or recall, AUC doesn't depend on picking any single threshold at
all: it measures whether the model *ranks* actual fraud cases higher than actual non-fraud cases, on
average, across every possible threshold at once. That's why it's a common first metric to check on an
imbalanced problem, before even deciding where to set the threshold.

---

## Part 5 — Real Data: Predicting Late NYC 311 Complaints

Same NYC 311 dataset as Lecture 7, reframed as a classification problem: instead of predicting the exact
`resolution_time_hours`, predict a binary outcome — **will this complaint take more than 3 days to
resolve?** Most complaints close quickly, so "late" is genuinely the rare class here, not a toy 10%
picked to make a point.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Built with a similar
    # right-skewed resolution-time shape to the real snapshot, so "late" (>72h) stays a genuine
    # minority class either way.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.05
    resolution_hours = rng.gamma(shape=1.5, scale=22.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

# The target for this lecture: did this complaint take more than 3 days (72 hours) to close?
complaints_df["is_late"] = (complaints_df["resolution_time_hours"] > 72).astype(int)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
print(f"'Late' (>72h) rate: {complaints_df['is_late'].mean():.1%}")

In [ ]:
majority_baseline = max(complaints_df["is_late"].mean(), 1 - complaints_df["is_late"].__________())
print(f"Majority-class baseline accuracy: {majority_baseline:.1%}")

### Split, preprocess, fit

Same workflow as Lectures 7 and 8: split first, then a `Pipeline` bundling the `ColumnTransformer` and the
model so preprocessing never sees the test set during fitting. Features: `hour_filed`, `borough`,
`complaint_grouped` — the same three that predicted `resolution_time_hours` reasonably well back in
Lecture 7.

In [ ]:
X = complaints_df[["hour_filed", "borough", "complaint_grouped"]]
y = complaints_df["is_late"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough", "complaint_grouped"]),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model.__________(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Training rows:", len(X_train), "| Test rows:", len(X_test))

In [ ]:
print(f"Accuracy: {(y_pred == y_test).mean():.3f}  (baseline: {majority_baseline:.3f})")
print()
print(classification_report(y_test, y_pred, target_names=["on time", "__________"]))

Accuracy likely lands close to the baseline — maybe even *at* it. Now look at recall for the "late"
row: on a real imbalanced problem, a plain `.predict()` at the default 0.5 threshold often catches only a
small fraction of the minority class, exactly like the toy fraud example in Part 1. **This is not a bug
in the code.** It's what happens when a model that's honestly weighing the evidence still isn't confident
enough, for most late complaints, to cross 0.5 — the same "correctly built model, weak result" lesson from
Lecture 7's regression R².

In [ ]:
cm_311 = __________(y_test, y_pred)
ConfusionMatrixDisplay(cm_311, display_labels=["on time", "late"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion Matrix — NYC 311 'Late' Classifier")
plt.show()

In [ ]:
for threshold in [0.5, 0.3, 0.2]:
    preds_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, preds_t, zero_division=0)
    r = recall_score(y_test, __________)
    print(f"threshold={threshold:.1f}: precision={p:.3f}, recall={r:.3f}")

Lowering the threshold trades the same way it did in the toy example: recall climbs as the model is
allowed to flag more complaints as "possibly late," precision falls as more of those flags turn out
wrong. Whether that trade is worth it is a City-operations question, not a math question: if flagging a
complaint "at risk of being late" just triggers a reminder email, low precision is cheap to tolerate. If
it triggers an expensive expedited crew, precision matters a lot more.

In [ ]:
fpr_311, tpr_311, _ = roc_curve(y_test, y_proba)
auc_311 = __________(y_test, y_proba)

plt.plot(fpr_311, tpr_311, color="#2B6CB0", linewidth=2, label=f"This model (AUC={auc_311:.3f})")
plt.plot([0, 1], [0, 1], color="#9AA5B1", linestyle="--", label="Random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — NYC 311 'Late' Classifier")
plt.legend()
plt.show()

Even where accuracy barely beats the baseline and a 0.5-threshold recall looks weak, AUC can still land
meaningfully above 0.5 — evidence the model is doing genuine ranking work (giving actually-late complaints
higher scores on average) even though the default threshold isn't the right place to read that off.
That's exactly why AUC is worth checking before writing off a model that "only" matches the baseline on
accuracy.

### Bringing it together

Same four ideas, toy data and real data agreed on all of them: accuracy alone hides how a model handles
the minority class; the confusion matrix and precision/recall show it directly; the 0.5 threshold is a
default, not a law; and AUC measures ranking quality independent of wherever you set that threshold. This
is exactly the toolkit Assignment 3 asks you to use on a fraud-or-churn dataset of your own — picking the
right metric for the cost of being wrong, not just reaching for accuracy because it's the first thing
`.score()` gives you.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect09_evaluation_metrics_exercise.ipynb`.

---

## Cheat Sheet

| Task | Code |
|---|---|
| Confusion matrix | `confusion_matrix(y_test, y_pred)` |
| Plot a confusion matrix | `ConfusionMatrixDisplay(cm, display_labels=[...]).plot()` |
| Precision / Recall / F1 | `precision_score(...)`, `recall_score(...)`, `f1_score(...)` |
| All three at once | `classification_report(y_test, y_pred, target_names=[...])` |
| Predicted probabilities | `model.predict_proba(X_test)[:, 1]` |
| Threshold a probability | `(y_proba >= threshold).astype(int)` |
| Precision/recall at every threshold | `precision_recall_curve(y_test, y_proba)` |
| ROC curve points | `roc_curve(y_test, y_proba)` |
| AUC | `roc_auc_score(y_test, y_proba)` |
| Majority-class baseline | `max(y.mean(), 1 - y.mean())` |

---

## Key Terms

- **Confusion matrix**: a 2x2 table (for binary classification) of True Positives, False Positives, False
  Negatives, and True Negatives — every prediction falls into exactly one cell.
- **Precision**: $TP / (TP + FP)$ — of everything flagged positive, how much actually was.
- **Recall (a.k.a. sensitivity, TPR)**: $TP / (TP + FN)$ — of everything actually positive, how much was
  caught.
- **F1 score**: the harmonic mean of precision and recall; punishes a large gap between them more than a
  plain average would.
- **Threshold**: the cutoff (default 0.5) applied to a predicted probability to produce a class label.
  Lowering it trades precision for recall.
- **False Positive Rate (FPR)**: $FP / (FP + TN)$ — of everything actually negative, how much was wrongly
  flagged positive.
- **ROC curve**: a plot of FPR (x) vs. TPR (y) as the threshold sweeps across all possible values.
- **AUC (area under the ROC curve)**: a threshold-independent measure of ranking quality; 1.0 is perfect,
  0.5 is random guessing.
- **Majority-class baseline**: the accuracy from always predicting the more common class — the bar any
  real classifier has to clear, and a number that alone can be as high as, or higher than, a real model's
  accuracy on an imbalanced problem.